In [3]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
sns.set_theme(color_codes=True)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.utils import resample

from xgboost import XGBClassifier, XGBRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV

In [4]:
pwd

'/mnt/Windows/Users/ashto/Disk/Projects/BoxOfficeAI/backend/notebooks'

In [5]:
df = pd.read_csv('../../data/movies.csv')
df.shape

(436482, 29)

In [6]:
df.isnull().mean() * 100

id                       0.000000
title                    0.000000
vote_average             0.000000
vote_count               0.000000
status                   0.000000
release_date             4.898942
revenue                  0.000000
runtime                  0.000000
adult                    0.000000
backdrop_path           57.547848
budget                   0.000000
homepage                87.449654
tconst                   0.000000
original_language        0.000000
original_title           0.000000
overview                 9.847600
popularity               0.000000
poster_path             17.394303
tagline                 78.878625
genres                  18.265816
production_companies    40.000504
production_countries    26.526867
spoken_languages        23.920803
keywords                60.894836
directors                2.278903
writers                 14.897980
averageRating            0.000000
numVotes                 0.000000
cast                    15.765369
dtype: float64

In [7]:
df2 = df.drop(columns=[
    "homepage","tagline","keywords","backdrop_path","id","tconst",
    "title","original_title","poster_path",
    "production_companies","production_countries","spoken_languages"
], errors="ignore")
df2[["overview","genres","cast","writers"]] = df2[["overview","genres","cast","writers"]].fillna("")
df2["directors"] = df2["directors"].fillna("Unknown")
df2 = df2.dropna(subset=["release_date"])
df2["release_year"] = pd.to_datetime(df2["release_date"]).dt.year

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df2["original_language"] = LabelEncoder().fit_transform(df2["original_language"])
df2["status"] = LabelEncoder().fit_transform(df2["status"])
df2["adult"] = df2["adult"].astype(int)

In [8]:
numeric_features = [
    "budget", "popularity", "runtime",
    "vote_average", "vote_count",
    "averageRating", "numVotes", "release_year"
]
text_feature = "overview"
categorical_features = ["status", "original_language", "adult"]
target_reg = "revenue"
target_class = (df2["revenue"] > df2["budget"] * 1.5).astype(int)

In [9]:
from scipy.stats import zscore

z = np.abs(zscore(df2["revenue"]))
df_filtered = df2[(z < 3)]

print("Before:", df2.shape)
print("After :", df_filtered.shape)

Before: (415099, 18)
After : (413007, 18)


In [10]:
df_filtered = df_filtered.reset_index(drop=True)
x_numeric = df_filtered[numeric_features + categorical_features].values

x_text = np.load("../embeddings/x_text.npy")
x_text = x_text[:len(df_filtered)]
x = np.hstack([x_numeric, x_text])

y_reg = df_filtered["revenue"]
y_class = (df_filtered["revenue"] > df_filtered["budget"] * 1.5).astype(int).values

In [11]:
# Uncomment if to be saved first time and not repeat the commands 

# import numpy as np

# np.save("x_text.npy", x_text)
# np.save("x.npy", x)
# np.save("y_reg.npy", y_reg.values)
# np.save("y_class.npy", y_class.values)

In [12]:
from sklearn.model_selection import train_test_split
budget = df_filtered["budget"].values

x_train, x_test, y_reg_train, y_reg_test, y_class_train, y_class_test, budget_train, budget_test = train_test_split(
    x,
    y_reg,
    y_class,
    budget,
    test_size=0.2,
    random_state=42
)

In [13]:
x_train = x_train.astype("float32")
x_test = x_test.astype("float32")

y_reg_train_log = np.log1p(y_reg_train + 1e5)
y_reg_test_log = np.log1p(y_reg_test + 1e5)

In [14]:
scale_pos_weight = (len(y_class_train) - sum(y_class_train)) / sum(y_class_train)

xgb_clf = XGBClassifier(
    tree_method="hist",
    device="cuda",
    n_estimators=600,
    max_depth=7,
    learning_rate=0.035,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=2,
    reg_lambda=3,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss"
)

cat_clf = CatBoostClassifier(
    iterations=300,
    depth=8,
    learning_rate=0.05,
    task_type="GPU",
    verbose=0
)

xgb_reg = XGBRegressor(
    tree_method="hist",
    device="cuda",
    n_estimators=300,
    max_depth=7,
    learning_rate=0.05
)

cat_reg = CatBoostRegressor(
    iterations=1200,
    depth=10,
    learning_rate=0.025,
    l2_leaf_reg=7,
    task_type="GPU",
    verbose=0
)

nb = GaussianNB()

models_clf = {
    "XGBoost": xgb_clf,
    "CatBoost": cat_clf,
    "Naive Bayes": nb
}

models_reg = {
    "XGBoost": xgb_reg,
    "CatBoost": cat_reg
}

In [15]:
df_train = pd.DataFrame(x_train)
df_train["target"] = y_class_train

df_majority = df_train[df_train["target"] == 0]
df_minority = df_train[df_train["target"] == 1]

df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df_balanced = pd.concat([df_majority, df_minority_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42)

x_train_res = df_balanced.drop("target", axis=1).values
y_train_res = df_balanced["target"].values

In [16]:
results_class = {}

for name, model in models_clf.items():
    print(f"\n🚀 Training {name} (Classification)...")

    if name == "XGBoost":
        model.fit(x_train, y_class_train)
    else:
        model.fit(x_train_res, y_train_res)

    y_pred = model.predict(x_test)   # ✅ FIXED (no clipping)

    results_class[name] = {
        "Accuracy": accuracy_score(y_class_test, y_pred),
        "Precision": precision_score(y_class_test, y_pred, zero_division=0),
        "Recall": recall_score(y_class_test, y_pred),
        "F1 Score": f1_score(y_class_test, y_pred)
    }

df_class = pd.DataFrame(results_class).T
display(df_class)


🚀 Training XGBoost (Classification)...


/home/subtilizer/miniconda/envs/ml/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [17:21:56] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



🚀 Training CatBoost (Classification)...

🚀 Training Naive Bayes (Classification)...


,Accuracy,Precision,Recall,F1 Score
XGBoost,0.942592,0.242180,0.650075,0.352893
CatBoost,0.899748,0.164963,0.778783,0.272256
Naive Bayes,0.970872,0.355509,0.257919,0.298951


In [ ]:
results_reg = {}

for name, model in models_reg.items():
    print(f"\n🚀 Training {name} (Regression)...")

    model.fit(x_train, y_reg_train_log)

    y_pred_log = model.predict(x_test)
    y_pred = np.expm1(y_pred_log) - 1e5
    y_pred = np.clip(y_pred, 0, None)

    results_reg[name] = {
        "MAE": mean_absolute_error(y_reg_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_reg_test, y_pred)),
        "R2_log": r2_score(y_reg_test_log, y_pred_log),
        "R2_real": r2_score(y_reg_test, y_pred)
    }

df_reg = pd.DataFrame(results_reg).T
display(df_reg)


🚀 Training XGBoost (Regression)...

🚀 Training CatBoost (Regression)...


In [ ]:
print("\n========== FINAL SUMMARY ==========")

print("\n🎯 Classification:")
display(df_class)

print("\n💰 Regression:")
display(df_reg)

In [ ]:
best_reg = models_reg["CatBoost"]

# train predictions
y_pred_train_log = best_reg.predict(x_train)
y_pred_train = np.expm1(y_pred_train_log)

# test predictions
y_pred_test_log = best_reg.predict(x_test)
y_pred_test = np.expm1(y_pred_test_log)

# 🔥 use ratio (better than raw revenue)
train_ratio = np.log1p(y_pred_train) - np.log1p(budget_train + 1)
test_ratio = np.log1p(y_pred_test) - np.log1p(budget_test + 1)

train_diff = y_pred_train - budget_train
test_diff = y_pred_test - budget_test

x_train_aug = np.hstack([
    x_train,
    train_ratio.reshape(-1, 1),
    train_diff.reshape(-1, 1)
])

x_test_aug = np.hstack([
    x_test,
    test_ratio.reshape(-1, 1),
    test_diff.reshape(-1, 1)
])

In [ ]:
best_clf = models_clf["XGBoost"] 

calibrated_clf = CalibratedClassifierCV(best_clf, method='sigmoid', cv=3)
calibrated_clf.fit(x_train_aug, y_class_train)

y_prob = calibrated_clf.predict_proba(x_test_aug)[:, 1]

In [ ]:
best_thresh = 0.5
best_score = 0

for t in np.arange(0.2, 0.5, 0.01):
    preds = (y_prob > t).astype(int)

    precision = precision_score(y_class_test, preds, zero_division=0)
    recall = recall_score(y_class_test, preds)

    score = (precision * recall)

    if score > best_score:
        best_score = score
        best_thresh = t

print("Better Threshold:", best_thresh)

In [ ]:
y_pred = (y_prob > best_thresh).astype(int)

In [ ]:
print("\n📊 FINAL CLASSIFICATION")

print("Accuracy :", accuracy_score(y_class_test, y_pred))
print("Precision:", precision_score(y_class_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_class_test, y_pred))
print("F1 Score :", f1_score(y_class_test, y_pred))

print("\n📊 FINAL REGRESSION")

print("MAE :", mean_absolute_error(y_reg_test, y_pred_test))
print("RMSE:", np.sqrt(mean_squared_error(y_reg_test, y_pred_test)))
print("R2  :", r2_score(y_reg_test, y_pred_test))

In [ ]:
import joblib

# save models
joblib.dump(calibrated_clf, "../model/clf_model.pkl")
joblib.dump(best_reg, "../model/reg_model.pkl")

# save any preprocessing artifacts if needed
joblib.dump(scale_pos_weight, "../model/meta.pkl")